In [33]:
import pandas as pd
import pandas as np

df = pd.read_csv("D:\\EcoPack AI\\data\\processed\\materials_featured.csv")

print(df.shape)
print(df.columns)

(22, 19)
Index(['material_id', 'material_type', 'strength_mpa', 'weight_capacity',
       'co2_emission_per_kg', 'biodegradability_score', 'recyclability_pct',
       'cost_inr_per_kg', 'material_category', 'strength_level', 'co2_norm',
       'cost_norm', 'strength_norm', 'emission_score', 'recyclability_index',
       'co2_impact_index', 'cost_score', 'cost_efficiency_index',
       'material_suitability_score'],
      dtype='str')


In [34]:
import sys
import os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(PROJECT_ROOT)

print(PROJECT_ROOT)

d:\EcoPack AI


In [35]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

In [36]:
# Paths
BASE_DIR = PROJECT_ROOT

DATA_PATH = os.path.join(
    BASE_DIR, "data", "processed", "materials_featured.csv"
)

MODELS_DIR = os.path.join(BASE_DIR, "models")
os.makedirs(MODELS_DIR, exist_ok=True)

In [37]:
# Load data
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

Dataset shape: (22, 19)
Columns: ['material_id', 'material_type', 'strength_mpa', 'weight_capacity', 'co2_emission_per_kg', 'biodegradability_score', 'recyclability_pct', 'cost_inr_per_kg', 'material_category', 'strength_level', 'co2_norm', 'cost_norm', 'strength_norm', 'emission_score', 'recyclability_index', 'co2_impact_index', 'cost_score', 'cost_efficiency_index', 'material_suitability_score']


In [38]:
# Feature Engineering (from notebook)
strength_mapping = {
    "Low": 1,
    "Medium": 2,
    "High": 3
}

df["strength_encoded"] = df["strength_level"].map(strength_mapping)


In [39]:
# Feature selection
X = df[
    [
        "strength_encoded",
        "weight_capacity",
        "biodegradability_score",
        "recyclability_pct",
        "cost_efficiency_index"
    ]
]

In [40]:
# Target variables
y_cost = df["cost_inr_per_kg"]
y_co2 = df["co2_impact_index"]

print("X shape:", X.shape)
print("y_cost shape:", y_cost.shape)
print("y_co2 shape:", y_co2.shape)

X shape: (22, 5)
y_cost shape: (22,)
y_co2 shape: (22,)


In [41]:
# Train-test split
X_train, X_test, y_cost_train, y_cost_test = train_test_split(
    X,
    y_cost,
    test_size=0.2,
    random_state=42
)

In [42]:
# Align CO2 target with same indices
y_co2_train = y_co2.loc[y_cost_train.index]
y_co2_test = y_co2.loc[y_cost_test.index]

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (17, 5)
Test shape: (5, 5)


In [ ]:
# -------------------------------------------------
# Cost Model – Random Forest
# -------------------------------------------------


cost_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

cost_model.fit(X_train, y_cost_train)

cost_predictions = cost_model.predict(X_test)

print("\nCOST MODEL EVALUATION")
print("MAE :", mean_absolute_error(y_cost_test, cost_predictions))
print("RMSE:", numpy.sqrt(mean_squared_error(y_cost_test, cost_predictions)))
print("R2  :", r2_score(y_cost_test, cost_predictions))



COST MODEL EVALUATION
MAE : 10.554000000000006


AttributeError: module 'pandas' has no attribute 'sqrt'

In [ ]:
# CO2 Model – XGBoost
co2_model = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)

co2_model.fit(X_train, y_co2_train)

co2_predictions = co2_model.predict(X_test)

print("\nCO2 MODEL EVALUATION")
print("MAE :", mean_absolute_error(y_co2_test, co2_predictions))
print("RMSE:", np.sqrt(mean_squared_error(y_co2_test, co2_predictions)))
print("R2  :", r2_score(y_co2_test, co2_predictions))



CO2 MODEL EVALUATION
MAE : 0.10224292483246118
RMSE: 0.19359427242000016
R2  : 0.2509337809832177


In [ ]:
# -------------------------------------------------
# CO2 Model – XGBoost
# -------------------------------------------------
co2_model = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)

co2_model.fit(X_train, y_co2_train)

co2_predictions = co2_model.predict(X_test)

print("\nCO2 MODEL EVALUATION")
print("MAE :", mean_absolute_error(y_co2_test, co2_predictions))
print("RMSE:", np.sqrt(mean_squared_error(y_co2_test, co2_predictions)))
print("R2  :", r2_score(y_co2_test, co2_predictions))



CO2 MODEL EVALUATION
MAE : 0.10224292483246118
RMSE: 0.19359427242000016
R2  : 0.2509337809832177


In [ ]:
import joblib


joblib.dump(cost_model, os.path.join(MODELS_DIR, "cost_model.pkl"))
joblib.dump(co2_model, os.path.join(MODELS_DIR, "co2_model.pkl"))

print("\nModels saved successfully in models/ folder")


Models saved successfully in models/ folder
